In [10]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
%autoreload 2

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier


from prepare_data import load_and_prepare
from model import DecisionTree

In [ ]:
X_train, X_test, y_train, y_test, feature_metadata, _ = load_and_prepare(csv_path="weatherAUS.csv")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=43, stratify=y_train
)

# Собственная реализация

In [28]:
feature_types = feature_metadata["feature_types"]

In [31]:
tree = DecisionTree(min_gain=1e-6, min_samples_leaf=5, max_depth=20)
tree.fit(X_tr, y_tr, feature_types=feature_types)

## Обучение без прунинга

In [ ]:
stats_before = tree.get_tree_stats()
print("Tree before pruning:", stats_before)

y_pred = tree.predict(X_test)
acc_before = accuracy_score(y_test, y_pred)
f1_before = f1_score(y_test, y_pred, average="weighted")
print("\nID3 before pruning:")
print("  Accuracy:", acc_before)
print("  F1 (weighted):", f1_before)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Tree before pruning: {'n_nodes': 7570, 'n_leaves': 5252, 'depth': 24}

ID3 before pruning:
  Accuracy: 0.8095572980765849
  F1 (weighted): 0.7883391272434641
Confusion matrix:
 [[20761  1303]
 [ 4113  2262]]


## Обучение с прунингом 

In [38]:
tree.prune(X_val.values, y_val)
stats_after = tree.get_tree_stats()
print("Tree after pruning:", stats_after)

y_pred_after = tree.predict(X_test)
acc_after = accuracy_score(y_test, y_pred_after)
f1_after = f1_score(y_test, y_pred_after, average="weighted")
print("\nID3 after pruning:")
print("  Accuracy:", acc_after)
print("  F1 (weighted):", f1_after)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_after))

Tree after pruning: {'n_nodes': 1758, 'n_leaves': 1329, 'depth': 22}

ID3 after pruning:
  Accuracy: 0.8285101445198495
  F1 (weighted): 0.8114675018360351
Confusion matrix:
 [[20917  1147]
 [ 3730  2645]]


# Реализация sklearn

In [44]:
sk_tree.tree_.n_leaves

np.int64(6175)

In [45]:
X_tr_fill = X_tr.fillna(X_tr.median())
X_test_fill = X_test.fillna(X_tr.median())
sk_tree = DecisionTreeClassifier(criterion="gini", random_state=42, min_samples_leaf=5, max_depth=20)
sk_tree.fit(X_tr_fill, y_tr)
y_sk = sk_tree.predict(X_test_fill)
acc_sk = accuracy_score(y_test, y_sk)
f1_sk = f1_score(y_test, y_sk, average="weighted")
n_nodes_sk = sk_tree.tree_.node_count
print("Sklearn tree nodes:", n_nodes_sk, "leaves:", sk_tree.tree_.n_leaves)
print("  Accuracy:", acc_sk)
print("  F1 (weighted):", f1_sk)
print("Confusion matrix:\n", confusion_matrix(y_test, y_sk))

Sklearn tree nodes: 12349 leaves: 6175
  Accuracy: 0.8105770245085974
  F1 (weighted): 0.8068455577804499
Confusion matrix:
 [[19697  2367]
 [ 3020  3355]]
